# Anomaly Detection Notebook

### Directories

In [18]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import joblib

### 1. LOAD DATA

In [19]:
df = pd.read_csv("cleaned_dataset_demo.csv")
label_col = "is_anomaly"

y = df[label_col].astype(int).values
X = df.drop(columns=[label_col])

for col in X.columns: # encode categorical variables
    if X[col].dtype == object:
        X[col] = X[col].astype('category').cat.codes

X = X.fillna(X.median()) # filling in missing values

X = X.select_dtypes(include=[np.number]) # keeping only numerical features

scaler = StandardScaler() # scale features
X_scaled = scaler.fit_transform(X)

### 2. TRAIN MODEL

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42, stratify=y)

clf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42
)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99     13528
           1       1.00      0.87      0.93      1472

    accuracy                           0.99     15000
   macro avg       0.99      0.93      0.96     15000
weighted avg       0.99      0.99      0.99     15000

ROC AUC: 0.9906360429174272
Confusion matrix:
 [[13528     0]
 [  192  1280]]


### 3. SAVE MODEL + SCALER

In [21]:
joblib.dump(clf, "anomaly_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(X.columns.tolist(), "model_features.pkl")

print("Model, scaler, and feature list saved successfully!")

Model, scaler, and feature list saved successfully!


### 4. FUNCTION TO PREDICT ANOMALY FOR ONE LOG

In [22]:
def predict_log(log_dict):
    """
    Input: one log in the same format as dataset columns
    Output: prediction (0 = normal, 1 = anomaly), probability
    """

    clf = joblib.load("anomaly_model.pkl")
    scaler = joblib.load("scaler.pkl")
    feature_cols = joblib.load("model_features.pkl")

    df_log = pd.DataFrame([log_dict])

    for col in df_log.columns:
        if df_log[col].dtype == object:
            df_log[col] = df_log[col].astype('category').cat.codes

    df_log = df_log.fillna(df_log.median())

    df_log = df_log.reindex(columns=feature_cols, fill_value=0)

    df_scaled = scaler.transform(df_log)

    pred = clf.predict(df_scaled)[0]
    proba = clf.predict_proba(df_scaled)[0][1]

    return {
        "anomaly_prediction": int(pred),
        "anomaly_probability": round(float(proba), 4)
    }


### 5. EXAMPLE: TEST WITH YOUR SAMPLE LOG

In [33]:
sample_log = {
    "index": 0,
    "timestamp": 7642205,
    "user_id": -4324475583306591935,
    "round_trip_time": None,
    "ip_address": "170.39.77.100",
    "country": "US",
    "region": "-",
    "city": "-",
    "asn": 393398,
    "user_agent": "Mozilla/5.0 (iPhone; CPU iPhone OS 14_2_1 like Mac OS X)",
    "browser_name": "Chrome Mobile 81.0.4044.1962",
    "os_name": "iOS 14.2.1",
    "device_type": "mobile",
    "login_successful": False,
    "is_attack_ip": False,
    "is_account_takeover": False,
    "hour": 9,
    "day": "2020-05-18",
    "day_of_week": "Monday"
}

print(predict_log(sample_log))

{'anomaly_prediction': 0, 'anomaly_probability': 0.34}
